# Week 2 — Alignment 

| | |
|---|---|
| **Due** | start of the Week 3 meeting |
| **Estimated time** | 3–4 hours
| **Prerequisites** | Week 1 |
| **Produces** | your hand-computed alignment; a working Needleman–Wunsch implementation (Computing ext.) |

> ### The hand exercise is done on paper, without an AI assistant.
> This is the only assignment all semester with that restriction. The value in this is the math passing through your own head. Every
> shortcut you meet for the rest of the course — BLAST, MMseqs2, Foldseek — is a
> shortcut around the matrix you are about to fill in by hand. You will spend the
> semester interpreting the output of tools built on this one algorithm.

## Learning objectives

1. Compute an optimal global alignment by hand using dynamic programming.
2. Explain what each element of an alignment score represents.
3. Justify a substitution matrix and gap penalty as claims about evolutionary distance.
4. Reproduce a hand result in software and reconcile any difference.
5. Distinguish global from local alignment and say when each is appropriate.

In [ ]:
#@title Setup — run this first
%pip install -q "biopython>=1.85" "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31"
print("setup complete")

In [ ]:
# Course data. 
import os, pathlib, requests

FIXTURES = pathlib.Path("fixtures")
COURSE_DATA_URL = 

def course_file(name: str) -> pathlib.Path:
    """Return a path to a course data file, downloading it if needed."""
    local = FIXTURES / name
    if local.exists():
        return local
    if COURSE_DATA_URL:
        FIXTURES.mkdir(exist_ok=True)
        r = requests.get(f"{COURSE_DATA_URL}/{name}", timeout=30)
        r.raise_for_status()
        local.write_bytes(r.content)
        return local
    raise FileNotFoundError(
        f"{name} not found. Put the fixtures folder beside this notebook, "
        "or set COURSE_DATA_URL."
    )

print("fixtures dir:", FIXTURES.resolve())

---
## Part 1: The hand exercise (on paper!)

Your two sequences are windows from the same two proteins in the class example, just a different region. The first is from *E. coli* DHFR (`P0ABQ4`), residues 37–45. The second is the corresponding region of **human** DHFR (`P00374`), residues 49–56.

```
A =  N K P V I M G R H       (E. coli DHFR 37-45,  9 residues)
B =  N L V I M G K K         (human DHFR  49-56,  8 residues)
```

Use **BLOSUM62** and a **linear gap penalty of d = 4** 

On paper:

1. Draw a grid 10 columns by 9 rows (one extra row and column for the initial gaps).
2. Fill the first row and column with 0, −4, −8, −12, …
3. Fill each remaining cell with the maximum of the three candidates: diagonal + BLOSUM62 score, up − 4, left − 4. **Record which one you chose** — draw the arrow. You will need the arrows for the traceback.
4. Trace back from the bottom-right cell to the top-left, following your arrows.
5. Write out the alignment and its score.

The BLOSUM62 values you will need will be printed after running the code chunk below. 

In [ ]:
from Bio.Align import substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")

A = "NKPVIMGRH"
B = "NLVIMGKK"

print("BLOSUM62 scores for every pair you might need:\n")
print("     " + "  ".join(f"{b:>3}" for b in B))
for a in A:
    row = "  ".join(f"{int(blosum62[a, b]):>3}" for b in B)
    print(f"  {a}  {row}")

### Record your answer

Fill in what you got **on paper**, before running anything else.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: three variables: my_score (int), my_top (str), my_bottom (str) — the two rows of your alignment with '-' for gaps
# ---- END ----


---
## Part 2: Now let the machine do it

Reconcile your paper result against Biopython. If they disagree, the interesting question is *where* — find the first cell where your matrix and its answer diverge.

In [ ]:
from Bio.Align import PairwiseAligner

aligner = PairwiseAligner()
aligner.substitution_matrix = blosum62
aligner.mode = "global"
aligner.open_gap_score = -4     # linear gap penalty: opening and extending
aligner.extend_gap_score = -4   # cost the same

result = aligner.align(A, B)
print("machine score:", result.score)
print("number of optimal alignments:", len(list(result)))
print()
print(result[0])

In [ ]:
print("your score   :", my_score)
print("machine score:", int(result.score))
print("agree?", my_score == int(result.score))

> **Warning** `PairwiseAligner()` has defaults, and they are
> not the defaults you want. If you forget to set `substitution_matrix`, it will silently score
> +1 for a match and 0 otherwise and give you a completely different number. Run the cell below to see it happen.

In [ ]:
naive = PairwiseAligner()          # no substitution matrix, no gap penalty
print("with defaults        :", naive.align(A, B).score)
print("with BLOSUM62, d=4   :", result.score)
print()
print("Same two sequences. The number means nothing without the scoring scheme.")

---
## Part 3: The gap penalty is a choice, not a setting

You used d = 4 because you were told to. What if it had been different?

Below, the same two sequences aligned across a range of gap penalties. Watch two things: the score, obviously, but also **the number of optimal alignments**.

In [ ]:
rows = []
for d in range(1, 13):
    al = PairwiseAligner()
    al.substitution_matrix = blosum62
    al.mode = "global"
    al.open_gap_score = -d
    al.extend_gap_score = -d
    r = al.align(A, B)
    rows.append((d, r.score, len(list(r))))

print(f"{'gap d':>6} {'score':>7} {'n optimal':>10}")
for d, s, n in rows:
    flag = "   <-- ambiguous" if n > 1 else ""
    print(f"{d:>6} {s:>7.0f} {n:>10}{flag}")

**Stop and think.** At d = 1 there are four equally optimal alignments. At d = 2 and above there is exactly one.

A cheap gap penalty makes gaps almost free, so the algorithm can insert them in several places for the same total score and has no basis for preferring one. The answer becomes ambiguous — not wrong, *ambiguous*, which is worse, because the software will still hand you a single alignment and say nothing about the other three.

This is the first appearance of a theme that will run to Week 13: **a tool that returns one answer is not telling you whether that answer was the only one.**

---
## Part 4: Global versus local

Everything so far has been *global* alignment: line up the sequences end to end, gaps and all. **Local** alignment finds the best-scoring subsegment instead and ignores the rest.

Same sequences, same matrix, same penalty. Only the mode changes.

In [ ]:
local = PairwiseAligner()
local.substitution_matrix = blosum62
local.mode = "local"
local.open_gap_score = -4
local.extend_gap_score = -4

lr = local.align(A, B)
print("local score :", lr.score, "  (global was", int(result.score), ")")
print()
print(lr[0])

The local alignment scores *higher* than the global one, and it is shorter. It has thrown away the flanking region (i.e., the part that did not align well) and kept only the conserved core.

That core is part of the DHFR active site, and it is recognisable between *E. coli* and human DHFR after roughly two billion years of divergence. The flanks are not conserved, and forcing them into an alignment costs you score without giving any additional information.

**When to use which.** Global when you believe the sequences are homologous along their whole length. Local when you are looking for a shared domain or motif inside otherwise unrelated sequences. BLAST, which you meet next week, is local...and now you know why.

---
## Extensions

### Computing extension: implement Needleman–Wunsch

The matrix initialisation and the traceback are written for you. **You write the scoring recurrence** — the three-way maximum in the middle. It is about three lines.

Then verify it reproduces both your hand answer and Biopython's.

In [ ]:
import numpy as np

def needleman_wunsch(seq_a, seq_b, matrix, d=4):
    """Global alignment by dynamic programming.

    Returns (score, aligned_a, aligned_b).
    """
    n, m = len(seq_a), len(seq_b)
    F = np.zeros((n + 1, m + 1))
    ptr = np.zeros((n + 1, m + 1), dtype=int)   # 0 diag, 1 up, 2 left

    # --- initialisation (given) ---
    for i in range(1, n + 1):
        F[i][0] = -d * i
        ptr[i][0] = 1
    for j in range(1, m + 1):
        F[0][j] = -d * j
        ptr[0][j] = 2

    # --- fill (YOURS) ---
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            pass  # replace me

    # --- traceback (given) ---
    a_out, b_out = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ptr[i][j] == 0:
            a_out.append(seq_a[i - 1]); b_out.append(seq_b[j - 1]); i -= 1; j -= 1
        elif i > 0 and ptr[i][j] == 1:
            a_out.append(seq_a[i - 1]); b_out.append("-"); i -= 1
        else:
            a_out.append("-"); b_out.append(seq_b[j - 1]); j -= 1
    return F[n][m], "".join(reversed(a_out)), "".join(reversed(b_out))

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: the three-way maximum inside the fill loop, plus the pointer that records which choice won
# Example: after your fix, needleman_wunsch(A, B, blosum62, 4) must return score 20.0
# ---- END ----


### Biology extension: the matrix is a claim about evolutionary distance

BLOSUM62 was built by counting substitutions in blocks of aligned protein segments that were at most 62% identical. BLOSUM45 used a 45% threshold. In other words, more distantly related sequences.

Redo the alignment with **BLOSUM45** and with at least two different gap penalties. Then answer in writing:

1. Which matrix would you choose for aligning two bacterial DHFRs? For a bacterial and a human one? Why?
2. Find a parameter combination that changes the alignment, not just the score. What changed, and which version do you believe?

A positive BLOSUM score does **not** mean "chemically similar". It means "observed in aligned blocks more often than chance would predict". Those are very different claims!

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: the BLOSUM45 comparison and your written answer
# Example: show at least two gap penalties, and state which alignment you believe and why
# ---- END ----


---
## Submission checklist

- [ ] Paper matrix and traceback, photographed or scanned, attached
- [ ] `my_score`, `my_top`, `my_bottom` filled in from your paper work
- [ ] Reconciliation against Biopython, with an account of any difference
- [ ] The gap-penalty sweep run, with your reading of the d = 1 result
- [ ] At least one extension, clearly labelled
- [ ] Notebook runs top to bottom in a fresh runtime


**Reminder:** the paper matrix and traceback are done without an AI assistant. If you used one for the Biopython reconciliation or an extension, note where and for what.